# Chapter 12: A Ship That Fires Bullets

> En este capítulo se empieza el juego **Alien Invasion** con Pygame: una nave que se mueve a izquierda y derecha y dispara balas.

## Summary

Alien Invasion se construye con **Pygame**, una colección de módulos para gráficos, animación y sonido.
El juego es una clase (`AlienInvasion`) con un **main loop** (`while True`) que en cada vuelta revisa los eventos del teclado y del mouse, actualiza las posiciones y redibuja la pantalla, a 60 vueltas por segundo gracias a un `Clock`.
Los elementos del juego (la nave, las balas) son clases que se tratan como **rects** (rectángulos): la nave se carga desde una imagen `.bmp` y se dibuja con `blit()`, y las balas se dibujan con `pygame.draw.rect()`.
Los valores del juego se guardan en una clase `Settings`, el movimiento continuo se logra con *flags* (`moving_right`, `moving_left`) que se activan con `KEYDOWN` y se desactivan con `KEYUP`, y las balas se guardan en un `Group` de Pygame.
Todo el tiempo se hace **refactoring**: dividir métodos largos en métodos auxiliares (*helper methods*, con `_` al inicio) para que el código siga siendo fácil de ampliar.

## Cheat Sheet

### Funciones, métodos y atributos de Pygame

| Quiero... | Sintaxis | Ejemplo | Resultado / nota |
|---|---|---|---|
| Instalar Pygame (terminal) | `python -m pip install --user pygame` | | si se usa `python3`, usar ese comando |
| Inicializar Pygame | `pygame.init()` | | prepara los ajustes internos que Pygame necesita |
| Crear la ventana | `pygame.display.set_mode((ancho, alto))` | `self.screen = pygame.display.set_mode((1200, 800))` | devuelve una *surface* que representa toda la ventana |
| Ventana en pantalla completa | `pygame.display.set_mode((0, 0), pygame.FULLSCREEN)` | | hay que poder salir con `Q` antes de usarla |
| Poner título a la ventana | `pygame.display.set_caption(texto)` | `pygame.display.set_caption("Alien Invasion")` | |
| Obtener los eventos | `pygame.event.get()` | `for event in pygame.event.get():` | lista de eventos desde la última llamada |
| Detectar cerrar la ventana | `event.type == pygame.QUIT` | | se responde con `sys.exit()` |
| Detectar tecla presionada | `event.type == pygame.KEYDOWN` | | |
| Detectar tecla soltada | `event.type == pygame.KEYUP` | | |
| Saber qué tecla fue | `event.key == pygame.K_RIGHT` | `pygame.K_LEFT`, `pygame.K_q`, `pygame.K_SPACE` | |
| Salir del programa | `sys.exit()` | | `import sys` |
| Mostrar lo último dibujado | `pygame.display.flip()` | | actualiza la pantalla en cada vuelta del loop |
| Crear un reloj | `pygame.time.Clock()` | `self.clock = pygame.time.Clock()` | |
| Controlar los cuadros por segundo | `clock.tick(fps)` | `self.clock.tick(60)` | una vez por vuelta del loop |
| Pintar el fondo | `screen.fill(color)` | `self.screen.fill((230, 230, 230))` | color RGB: `(rojo, verde, azul)` de 0 a 255 |
| Cargar una imagen | `pygame.image.load(ruta)` | `pygame.image.load('images/ship.bmp')` | devuelve una *surface*; lo más simple es usar `.bmp` |
| Obtener el rect | `superficie.get_rect()` | `self.image.get_rect()` | |
| Dibujar una imagen | `screen.blit(imagen, rect)` | `self.screen.blit(self.image, self.rect)` | |
| Crear un rect desde cero | `pygame.Rect(x, y, ancho, alto)` | `pygame.Rect(0, 0, 3, 15)` | `x`, `y` es la esquina superior izquierda |
| Dibujar un rectángulo | `pygame.draw.rect(superficie, color, rect)` | `pygame.draw.rect(self.screen, self.color, self.rect)` | |
| Heredar de `Sprite` | `class Bullet(Sprite):` + `super().__init__()` | `from pygame.sprite import Sprite` | permite agrupar elementos |
| Crear un grupo | `pygame.sprite.Group()` | `self.bullets = pygame.sprite.Group()` | se comporta como una lista con funciones extra |
| Agregar a un grupo | `grupo.add(sprite)` | `self.bullets.add(new_bullet)` | equivale a `append()` |
| Actualizar todo el grupo | `grupo.update()` | `self.bullets.update()` | llama a `update()` de cada sprite |
| Lista de los sprites | `grupo.sprites()` | `for bullet in self.bullets.sprites():` | |
| Copiar un grupo | `grupo.copy()` | `for bullet in self.bullets.copy():` | para poder borrar elementos dentro del `for` |
| Quitar de un grupo | `grupo.remove(sprite)` | `self.bullets.remove(bullet)` | |
| Cantidad de elementos | `len(grupo)` | `len(self.bullets)` | |

### Atributos de un rect

| Para... | Atributos |
|---|---|
| Esquinas y bordes | `top`, `bottom`, `left`, `right` |
| Centro | `center`, `centerx`, `centery` |
| Puntos medios de un borde | `midtop`, `midbottom`, `midleft`, `midright` |
| Posición de la esquina superior izquierda | `x`, `y` |

Los atributos de un rect solo guardan **enteros**: para velocidades decimales hay que guardar la posición en un `float` aparte (`self.x`, `self.y`).

### Python que aparece en el capítulo

| Concepto | Ejemplo |
|---|---|
| Ejecutar solo si se llama al archivo directamente | `if __name__ == '__main__':` |
| Método auxiliar (*helper method*): lleva `_` al inicio | `def _check_events(self):` |
| Llamar a un método de la misma clase | `self._check_events()` |
| Convertir a decimal | `self.x = float(self.rect.x)` |

### Problemas del capítulo

| Problema | Mensaje | Causa | Solución |
|---|---|---|---|
| Las balas siguen existiendo fuera de la pantalla | (no hay mensaje) | su `y` se vuelve cada vez más negativo y siguen gastando memoria | quitarlas con `remove()` cuando `bullet.rect.bottom <= 0` |
| Modificar un grupo dentro del `for` | (no hay mensaje) | Python espera que la lista no cambie de largo mientras se recorre | recorrer una copia: `self.bullets.copy()` |
| El juego se vuelve lento | (no hay mensaje) | quedó un `print()` dentro del loop | borrar el `print()` después de probar |
| La nave se sale de la pantalla | (no hay mensaje) | no se revisa el borde antes de mover | comparar `rect.right` y `rect.left` con los bordes de la pantalla |
| No se puede salir en pantalla completa | (no hay mensaje) | Pygame no trae una forma de salir en `FULLSCREEN` | agregar antes la tecla `Q` para salir |

## Planning Your Project

Cuando se construye un proyecto grande, conviene preparar un plan antes de escribir código.
Una descripción general de **Alien Invasion**:

> El jugador controla una nave que aparece en la parte inferior central de la pantalla. La nave se mueve a derecha e izquierda con las flechas y dispara balas con la barra espaciadora. Al empezar, una flota de aliens llena el cielo y se mueve de lado a lado y hacia abajo. El jugador dispara y destruye a los aliens. Si destruye a todos, aparece una flota nueva más rápida. Si un alien toca la nave o llega al borde inferior, el jugador pierde una nave. Si pierde tres naves, el juego termina.

En la primera fase se hace una nave que se mueve con las flechas y dispara con la barra espaciadora. Después se crean los aliens.

> Alien Invasion usa varios archivos: hay que crear una carpeta `alien_invasion` y guardar ahí todos los archivos del proyecto, para que los `import` funcionen.

## Installing Pygame

Pygame se instala con `pip`, igual que `pytest` en el Capítulo 11. En la terminal:

## Starting the Game Project

Se empieza con una ventana de Pygame vacía. Después se dibujan en ella los elementos del juego, se responde a lo que hace el usuario, se pone un color de fondo y se carga la imagen de la nave.

### Creating a Pygame Window and Responding to User Input

La ventana se crea con una clase que representa el juego. Se guarda en un archivo llamado `alien_invasion.py`:

In [1]:
# alien_invasion.py
import sys

import pygame

class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()

        self.screen = pygame.display.set_mode((1200, 800))
        pygame.display.set_caption("Alien Invasion")

    def run_game(self):
        """Start the main loop for the game."""
        while True:
            # Watch for keyboard and mouse events.
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    sys.exit()

            # Make the most recently drawn screen visible.
            pygame.display.flip()

if __name__ == '__main__':
    # Make a game instance, and run the game.
    ai = AlienInvasion()
    ai.run_game()

ModuleNotFoundError: No module named 'pygame'

- Se importan `sys` (para salir del juego cuando el jugador cierre la ventana) y `pygame` (con todo lo necesario para hacer el juego).
- En `__init__()`, `pygame.init()` inicializa los ajustes que Pygame necesita y `pygame.display.set_mode()` crea la ventana. El argumento `(1200, 800)` es una tupla con el ancho y el alto en píxeles.
- El objeto que se guarda en `self.screen` es una **surface**: una parte de la pantalla donde se puede mostrar un elemento del juego. Cada elemento (un alien, la nave) es su propia surface. La que devuelve `set_mode()` representa toda la ventana y se redibuja en cada vuelta del loop.
- El juego se controla con `run_game()`, que tiene un `while True` (el **main loop**). Dentro hay un **event loop**: un `for` sobre `pygame.event.get()`, que devuelve la lista de eventos ocurridos desde la última vez que se llamó. Un **evento** es una acción del usuario, como presionar una tecla o mover el mouse.
- Cuando el jugador cierra la ventana se detecta un evento `pygame.QUIT` y se llama a `sys.exit()`.
- `pygame.display.flip()` hace visible la pantalla dibujada más recientemente. Al mover los elementos, `flip()` muestra las posiciones nuevas y oculta las viejas, creando la ilusión de movimiento suave.
- Al final del archivo se crea una instancia del juego y se llama a `run_game()` dentro de un `if __name__ == '__main__':`, para que solo se ejecute si el archivo se corre directamente.

### Controlling the Frame Rate

Idealmente el juego debería correr a la misma velocidad (**frame rate**) en todos los equipos.
Pygame lo resuelve con un reloj: se crea en `__init__()` y se hace avanzar (*tick*) una vez en cada vuelta del loop. Si el loop va más rápido que la velocidad definida, Pygame calcula cuánto debe pausar.

In [ ]:
# alien_invasion.py
    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.clock = pygame.time.Clock()
        # --snip--

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            # --snip--
            pygame.display.flip()
            self.clock.tick(60)

El método `tick()` recibe un argumento: los cuadros por segundo del juego. Con `60`, Pygame intenta que el loop corra exactamente 60 veces por segundo.
Si el reloj no funciona bien en el equipo, se puede probar otro valor o quitarlo y ajustar los *settings* para que el juego corra bien.

### Setting the Background Color

Pygame crea una pantalla negra por defecto. El color de fondo se define al final de `__init__()`.
Los colores son **RGB**: una mezcla de rojo, verde y azul, cada uno de 0 a 255. Por ejemplo, `(255, 0, 0)` es rojo, `(0, 255, 0)` es verde y `(0, 0, 255)` es azul. El color `(230, 230, 230)` tiene la misma cantidad de los tres, y da un gris claro.

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        pygame.display.set_caption("Alien Invasion")

        # Set the background color.
        self.bg_color = (230, 230, 230)

    def run_game(self):
        # --snip--
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    sys.exit()

            # Redraw the screen during each pass through the loop.
            self.screen.fill(self.bg_color)

            # Make the most recently drawn screen visible.
            pygame.display.flip()
            self.clock.tick(60)

El método `fill()` actúa sobre una surface y recibe un solo argumento: un color.

### Creating a Settings Class

Cada vez que se agrega una función nueva al juego, normalmente se agregan también nuevos *settings*.
En vez de repartirlos por todo el código, se escribe un módulo `settings` con una clase `Settings` que los guarda todos en un solo lugar. Para cambiar el juego, solo se modifican los valores de `settings.py`.

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        """Initialize the game's settings."""
        # Screen settings
        self.screen_width = 1200
        self.screen_height = 800
        self.bg_color = (230, 230, 230)

Luego, en `alien_invasion.py`, se crea una instancia de `Settings` y se usa para acceder a cada valor:

In [ ]:
# alien_invasion.py
# --snip--
import pygame

from settings import Settings

class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.clock = pygame.time.Clock()
        self.settings = Settings()

        self.screen = pygame.display.set_mode(
            (self.settings.screen_width, self.settings.screen_height))
        pygame.display.set_caption("Alien Invasion")

    def run_game(self):
            # --snip--
            # Redraw the screen during each pass through the loop.
            self.screen.fill(self.settings.bg_color)

            # Make the most recently drawn screen visible.
            pygame.display.flip()
            self.clock.tick(60)

# --snip--

Al ejecutarlo no se ve ningún cambio, porque solo se movieron los valores que ya se usaban. Ahora se pueden empezar a agregar elementos a la pantalla.

### Comparison: Starting the Game Project

| | `__init__()` | `run_game()` |
|---|---|---|
| Cuándo se ejecuta | **una vez**, al crear el juego | **muchas veces por segundo**, dentro del `while True` |
| Qué contiene | `pygame.init()`, el reloj, los `Settings`, la ventana y el título | el event loop, el `fill()`, el `flip()` y el `tick()` |
| Ejemplo | `self.screen = pygame.display.set_mode(...)` | `self.screen.fill(self.settings.bg_color)` |

| Sin | Con |
|---|---|
| `Clock` | la velocidad del juego depende de qué tan rápido sea el equipo |
| `self.clock.tick(60)` | el loop corre a 60 vueltas por segundo en cualquier equipo |
| `Settings` | los valores están repartidos por todo el código |
| `Settings` (`self.settings.bg_color`) | todos los valores están en un solo archivo, `settings.py` |

| Color | RGB |
|---|---|
| Rojo | `(255, 0, 0)` |
| Verde | `(0, 255, 0)` |
| Azul | `(0, 0, 255)` |
| Gris claro (fondo del juego) | `(230, 230, 230)` |

## Adding the Ship Image

Para dibujar la nave se carga una imagen y luego se usa el método `blit()`.
Conviene usar imágenes `.bmp` (bitmap), porque Pygame las carga por defecto. Si la imagen es `.jpg` o `.png`, se puede convertir con programas como GIMP.
También hay que fijarse en el **color de fondo** de la imagen: debería coincidir con el color de fondo del juego (o ser transparente), para que se vea bien.
Y en las licencias: al elegir imágenes, lo más seguro es usar gráficos de licencia libre, por ejemplo de https://opengameart.org.

Para Alien Invasion se usa `ship.bmp`, que está en los recursos del libro (https://ehmatthes.github.io/pcc_3e). Se guarda en una carpeta `images` dentro de la carpeta del proyecto.

### Creating the Ship Class

Se crea un módulo `ship` con la clase `Ship`, que maneja la mayor parte del comportamiento de la nave:

In [ ]:
# ship.py
import pygame

class Ship:
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        self.screen = ai_game.screen
        self.screen_rect = ai_game.screen.get_rect()

        # Load the ship image and get its rect.
        self.image = pygame.image.load('images/ship.bmp')
        self.rect = self.image.get_rect()

        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

    def blitme(self):
        """Draw the ship at its current location."""
        self.screen.blit(self.image, self.rect)

ModuleNotFoundError: No module named 'pygame'

- Pygame es eficiente porque permite tratar todos los elementos del juego como **rects** (rectángulos), aunque no tengan forma de rectángulo. Así, por ejemplo, detectar si dos elementos chocan es más rápido.
- `__init__()` recibe `self` y una referencia a la instancia actual de `AlienInvasion` (`ai_game`), lo que le da acceso a todos los recursos del juego. Se guarda la pantalla en `self.screen` y su rect en `self.screen_rect`, para poder ubicar la nave.
- `pygame.image.load()` carga la imagen y devuelve una surface (`self.image`). Con `get_rect()` se obtiene su `rect`, que se usa para ubicarla.
- Con un rect se pueden usar las coordenadas `x` e `y` de los bordes y del centro. Para centrar un elemento se usan `center`, `centerx` o `centery`; para trabajar en un borde, `top`, `bottom`, `left` o `right`; y hay combinaciones como `midbottom`, `midtop`, `midleft` y `midright`. Los atributos `x` e `y` son la esquina superior izquierda.
- `blitme()` dibuja la imagen en la posición que indica `self.rect`.

> En Pygame, el origen `(0, 0)` está en la esquina **superior izquierda** de la pantalla, y las coordenadas aumentan hacia abajo y hacia la derecha. En una pantalla de 1200 × 800, la esquina inferior derecha es `(1200, 800)`.

La nave se ubica abajo al centro: `self.rect.midbottom` se iguala al `midbottom` de la pantalla.

### Drawing the Ship to the Screen

En `alien_invasion.py` se crea la nave y se llama a `blitme()`:

In [ ]:
# alien_invasion.py
# --snip--
from settings import Settings
from ship import Ship

class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        # --snip--
        pygame.display.set_caption("Alien Invasion")

        self.ship = Ship(self)

    def run_game(self):
            # --snip--
            # Redraw the screen during each pass through the loop.
            self.screen.fill(self.settings.bg_color)
            self.ship.blitme()

            # Make the most recently drawn screen visible.
            pygame.display.flip()
            self.clock.tick(60)

# --snip--

`Ship(self)` recibe como argumento la instancia actual de `AlienInvasion`, que es lo que le da acceso a recursos como la pantalla.
La nave se dibuja **después** de llenar el fondo, para que aparezca encima.
Al ejecutarlo se ve la pantalla vacía con la nave abajo al centro.

### Comparison: Adding the Ship Image

| | `self.image` | `self.rect` |
|---|---|---|
| Qué es | la *surface* con la imagen cargada | el rectángulo que ocupa la imagen |
| Cómo se obtiene | `pygame.image.load('images/ship.bmp')` | `self.image.get_rect()` |
| Para qué sirve | es lo que se dibuja | ubicar y mover la imagen |

Con una pantalla de 1200 × 800:

| Atributo del rect | Qué es | Se usa para |
|---|---|---|
| `x`, `y` | esquina superior izquierda | ajustar la posición horizontal o vertical |
| `top`, `bottom`, `left`, `right` | bordes | trabajar en un borde de la pantalla |
| `center`, `centerx`, `centery` | centro | centrar un elemento |
| `midbottom`, `midtop`, `midleft`, `midright` | punto medio de un borde | ubicar contra un borde y centrado a la vez |

| | Se dibuja con | Se usa en |
|---|---|---|
| Imagen | `screen.blit(imagen, rect)` | la nave (`Ship`) |
| Rectángulo | `pygame.draw.rect(superficie, color, rect)` | las balas (`Bullet`) |

## Refactoring: The _check_events() and _update_screen() Methods

En proyectos grandes se hace **refactoring** antes de agregar más código: simplificar la estructura del código ya escrito para que sea más fácil ampliarlo.
Aquí se divide `run_game()`, que se está alargando, en dos **helper methods**: un método que hace trabajo dentro de la clase pero no se usa desde fuera. En Python, un guion bajo al inicio (`_`) indica un helper method.

### The _check_events() Method

El código que maneja los eventos se mueve a un método aparte:

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()

            # Redraw the screen during each pass through the loop.
            # --snip--
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()

Para llamar a un método desde dentro de la misma clase se usa la notación de punto con `self`: `self._check_events()`.

### The _update_screen() Method

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            self._update_screen()
            self.clock.tick(60)

    def _check_events(self):
        # --snip--
    def _update_screen(self):
        """Update images on the screen, and flip to the new screen."""
        self.screen.fill(self.settings.bg_color)
        self.ship.blitme()

        pygame.display.flip()

Ahora el cuerpo del loop principal es mucho más simple: se revisan los eventos, se actualiza la pantalla y se avanza el reloj.
Es un proceso realista de desarrollo: primero se escribe el código de la forma más simple posible, y se hace refactoring a medida que el proyecto se complica.

### Comparison: Refactoring

| | `run_game()` antes | `run_game()` después |
|---|---|---|
| Qué contiene | el event loop, el `fill()`, el `blitme()` y el `flip()` dentro del `while` | solo tres llamadas: `_check_events()`, `_update_screen()` y `clock.tick(60)` |
| Legibilidad | hay que leer todo para saber qué pasa | los nombres de los métodos explican qué ocurre |
| Dónde está el manejo de eventos | dentro del loop | en `_check_events()` |
| Dónde está el dibujo | dentro del loop | en `_update_screen()` |

### Try it Yourself

> 12-1. Blue Sky:
Make a Pygame window with a blue background.

> 12-2. Game Character:
Find a bitmap image of a game character you like or convert an image to a bitmap. Make a class that draws the character at the center of the screen, then match the background color of the image to the background color of the screen or vice versa.

In [ ]:
# 12-1


In [ ]:
# 12-2


## Piloting the Ship

Ahora el jugador podrá mover la nave a derecha e izquierda con las flechas. Primero se trabaja el movimiento a la derecha, y después se aplica lo mismo a la izquierda.

### Responding to a Keypress

Cada vez que el jugador presiona una tecla, Pygame lo registra como un evento, y `pygame.event.get()` lo recoge. Cada tecla presionada es un evento `KEYDOWN`.
Cuando se detecta un `KEYDOWN`, hay que revisar si la tecla es una que activa una acción:

In [ ]:
# alien_invasion.py
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RIGHT:
                    # Move the ship to the right.
                    self.ship.rect.x += 1

`event.key` es la tecla presionada, y la flecha derecha se representa con `pygame.K_RIGHT`. Si es esa tecla, la nave se mueve aumentando `self.ship.rect.x` en 1.
Con este código la nave se mueve un píxel por cada vez que se presiona la tecla. Funciona, pero no es una forma eficiente de controlarla.

### Allowing Continuous Movement

Para que la nave siga moviéndose mientras la tecla esté presionada, se usan los eventos `KEYDOWN` y `KEYUP` junto con un **flag** llamado `moving_right`:

- Mientras `moving_right` es `False`, la nave está quieta.
- Al presionar la flecha derecha, el flag pasa a `True`.
- Al soltarla, el flag vuelve a `False`.

La clase `Ship` controla los atributos de la nave, así que ahí se agrega el atributo `moving_right` y un método `update()` que cambia la posición si el flag es `True`. Este método se llama una vez en cada vuelta del `while`.

In [ ]:
# ship.py
class Ship:
    """A class to manage the ship."""

    def __init__(self, ai_game):
        # --snip--
        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

        # Movement flag; start with a ship that's not moving.
        self.moving_right = False

    def update(self):
        """Update the ship's position based on the movement flag."""
        if self.moving_right:
            self.rect.x += 1

    def blitme(self):
        # --snip--

`update()` se llama desde fuera de la clase, así que no es un helper method (no lleva `_`). Ahora `_check_events()` solo cambia el flag:

In [ ]:
# alien_invasion.py
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            # --snip--
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RIGHT:
                    self.ship.moving_right = True
            elif event.type == pygame.KEYUP:
                if event.key == pygame.K_RIGHT:
                    self.ship.moving_right = False

Y en `run_game()` se llama a `update()` en cada vuelta, **después** de revisar los eventos y **antes** de actualizar la pantalla:

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            self.ship.update()
            self._update_screen()
            self.clock.tick(60)

Así la posición se actualiza según lo que hizo el jugador y se usa la posición nueva al dibujar. Al mantener presionada la flecha derecha, la nave se mueve de forma continua.

### Moving Both Left and Right

Para el movimiento a la izquierda se repite lo mismo: otro flag (`moving_left`) y dos cambios en `_check_events()`.

In [ ]:
# ship.py
    def __init__(self, ai_game):
        # --snip--
        # Movement flags; start with a ship that's not moving.
        self.moving_right = False
        self.moving_left = False

    def update(self):
        """Update the ship's position based on movement flags."""
        if self.moving_right:
            self.rect.x += 1
        if self.moving_left:
            self.rect.x -= 1

En `update()` se usan **dos `if` separados** en vez de un `elif`. Así, si el jugador presiona ambas flechas a la vez, `rect.x` aumenta y luego disminuye, y la nave queda quieta. Con un `elif`, la flecha derecha siempre tendría prioridad.

In [ ]:
# alien_invasion.py
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            # --snip--
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_RIGHT:
                    self.ship.moving_right = True
                elif event.key == pygame.K_LEFT:
                    self.ship.moving_left = True
            elif event.type == pygame.KEYUP:
                if event.key == pygame.K_RIGHT:
                    self.ship.moving_right = False
                elif event.key == pygame.K_LEFT:
                    self.ship.moving_left = False

Aquí sí se puede usar `elif`, porque cada evento está asociado a una sola tecla. Si se presionan las dos teclas, se detectan dos eventos distintos.

### Adjusting the Ship's Speed

Ahora la nave se mueve 1 píxel por vuelta del loop. Para controlar mejor la velocidad se agrega un atributo `ship_speed` en `Settings`:

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        # --snip--
        # Ship settings
        self.ship_speed = 1.5

Con `1.5`, la posición cambia en 1,5 píxeles por vuelta. Se usa un decimal (`float`) para tener un control más fino cuando más adelante se aumente la velocidad del juego.
Pero los atributos de un rect, como `x`, solo guardan enteros (guardan solo la parte entera del número). Por eso `Ship` guarda la posición exacta en su propio atributo `self.x`:

In [ ]:
# ship.py
class Ship:
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        # --snip--
        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

        # Store a float for the ship's exact horizontal position.
        self.x = float(self.rect.x)

        # Movement flags; start with a ship that's not moving.
        self.moving_right = False
        self.moving_left = False

    def update(self):
        """Update the ship's position based on movement flags."""
        # Update the ship's x value, not the rect.
        if self.moving_right:
            self.x += self.settings.ship_speed
        if self.moving_left:
            self.x -= self.settings.ship_speed

        # Update rect object from self.x.
        self.rect.x = self.x

    def blitme(self):
        # --snip--

- Se crea `self.settings` para poder usarlo en `update()`.
- `float(self.rect.x)` convierte la posición a decimal y la guarda en `self.x`.
- `update()` cambia `self.x` según `ship_speed`, y luego usa ese valor para actualizar `self.rect.x`, que es lo que controla dónde se dibuja la nave. A `rect.x` solo llega la parte entera de `self.x`, y para dibujar es suficiente.

Ahora cualquier valor de `ship_speed` mayor que 1 hace que la nave se mueva más rápido.

### Limiting the Ship's Range

Si se mantiene presionada una flecha, la nave desaparece por el borde. Para que se detenga en el borde de la pantalla, se revisa su posición antes de cambiar `self.x`:

In [ ]:
# ship.py
    def update(self):
        """Update the ship's position based on movement flags."""
        # Update the ship's x value, not the rect.
        if self.moving_right and self.rect.right < self.screen_rect.right:
            self.x += self.settings.ship_speed
        if self.moving_left and self.rect.left > 0:
            self.x -= self.settings.ship_speed

        # Update rect object from self.x.
        self.rect.x = self.x

- `self.rect.right` es la coordenada x del borde derecho de la nave. Si es menor que `self.screen_rect.right`, la nave aún no llegó al borde derecho.
- `self.rect.left` es la coordenada x del borde izquierdo. Si es mayor que `0`, aún no llegó al borde izquierdo.

Con solo agregar una condición en un `if`, la nave se detiene como si chocara con una pared.

### Refactoring _check_events()

`_check_events()` va a seguir creciendo, así que se divide en dos métodos: uno para los `KEYDOWN` y otro para los `KEYUP`.

In [ ]:
# alien_invasion.py
    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                self._check_keydown_events(event)
            elif event.type == pygame.KEYUP:
                self._check_keyup_events(event)

    def _check_keydown_events(self, event):
        """Respond to keypresses."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = True
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = True

    def _check_keyup_events(self, event):
        """Respond to key releases."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = False
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = False

Cada método nuevo necesita los parámetros `self` y `event`. Sus cuerpos se copiaron de `_check_events()`, que ahora queda más simple.

### Pressing Q to Quit

Cerrar la ventana con el mouse cada vez que se prueba algo es tedioso, así que se agrega la tecla `Q` para terminar el juego:

In [ ]:
# alien_invasion.py
    def _check_keydown_events(self, event):
        # --snip--
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = True
        elif event.key == pygame.K_q:
            sys.exit()

### Running the Game in Fullscreen Mode

Pygame tiene un modo de pantalla completa. Se cambia en `__init__()`:

In [ ]:
# alien_invasion.py
    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.settings = Settings()

        self.screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
        self.settings.screen_width = self.screen.get_rect().width
        self.settings.screen_height = self.screen.get_rect().height
        pygame.display.set_caption("Alien Invasion")

Se pasa el tamaño `(0, 0)` y `pygame.FULLSCREEN`: así Pygame calcula un tamaño de ventana que llene la pantalla. Como no se conoce de antemano el ancho y el alto, se actualizan los *settings* después de crear la pantalla, con los atributos `width` y `height` del rect de la pantalla.
Si no gusta el modo de pantalla completa, se puede volver al tamaño fijo original.

> Hay que asegurarse de poder salir con `Q` antes de ejecutar en pantalla completa: Pygame no ofrece otra forma de salir en ese modo.

### Comparison: Piloting the Ship

| | Mover `rect.x += 1` en el `KEYDOWN` | Usar un flag (`moving_right`) |
|---|---|---|
| Qué hace al presionar la tecla | mueve 1 píxel, una sola vez | cambia el flag a `True` |
| Movimiento continuo | No, hay que presionar la tecla muchas veces | Sí, `update()` mueve la nave en cada vuelta |
| Al soltar la tecla | no pasa nada | `KEYUP` cambia el flag a `False` |

| | `KEYDOWN` | `KEYUP` |
|---|---|---|
| Cuándo ocurre | se presiona una tecla | se suelta una tecla |
| Con la flecha derecha | `moving_right = True` | `moving_right = False` |
| Método | `_check_keydown_events()` | `_check_keyup_events()` |

| | Dos `if` en `update()` | `if` + `elif` en `update()` |
|---|---|---|
| Con ambas flechas presionadas | la nave queda quieta (suma y resta) | la flecha derecha siempre gana |
| Cuándo usarlo | el movimiento depende de dos flags independientes | cada evento está asociado a una sola tecla (en `_check_keydown_events()`) |

| | `self.rect.x` | `self.x` |
|---|---|---|
| Tipo | entero (solo guarda la parte entera) | `float` |
| Para qué sirve | decide dónde se dibuja la nave | guarda la posición exacta |
| Se actualiza | al final de `update()`, con `self.rect.x = self.x` | con `+= self.settings.ship_speed` |

| | Ventana normal | Pantalla completa |
|---|---|---|
| Sintaxis | `set_mode((1200, 800))` | `set_mode((0, 0), pygame.FULLSCREEN)` |
| Ancho y alto | se definen en `Settings` | se leen de la pantalla con `get_rect().width` y `.height` |
| Salir | cerrar la ventana o `Q` | solo `Q` |

## A Quick Recap

Antes de agregar las balas, se repasa cómo está organizado el proyecto, que tiene tres archivos con varias clases y métodos.

**alien_invasion.py**
- Contiene la clase `AlienInvasion`, que crea los atributos importantes del juego: los *settings* (`settings`), la superficie principal (`screen`) y la instancia de la nave (`ship`).
- Aquí está el main loop (un `while`), que llama a `_check_events()`, `ship.update()` y `_update_screen()`, y avanza el reloj en cada vuelta.
- `_check_events()` detecta los eventos importantes y los procesa con `_check_keydown_events()` y `_check_keyup_events()`.
- `_update_screen()` redibuja la pantalla en cada vuelta.
- Es el **único archivo que se ejecuta** para jugar; los demás se importan.

**settings.py**
- Contiene la clase `Settings`, que por ahora solo tiene `__init__()` con los atributos de la apariencia del juego y la velocidad de la nave.

**ship.py**
- Contiene la clase `Ship`, con `__init__()`, `update()` (maneja la posición de la nave) y `blitme()` (la dibuja).
- La imagen está en `ship.bmp`, en la carpeta `images`.

### Try it Yourself

> 12-3. Pygame Documentation:
We’re far enough into the game now that you might want to look at some of the Pygame documentation. The Pygame home page is at https://pygame.org, and the home page for the documentation is at https://pygame.org/docs. Just skim the documentation for now. You won’t need it to complete this project, but it will help if you want to modify Alien Invasion or make your own game afterward.

> 12-4. Rocket:
Make a game that begins with a rocket in the center of the screen. Allow the player to move the rocket up, down, left, or right using the four arrow keys. Make sure the rocket never moves beyond any edge of the screen.

> 12-5. Keys:
Make a Pygame file that creates an empty screen. In the event loop, print the event.key attribute whenever a pygame.KEYDOWN event is detected. Run the program and press various keys to see how Pygame responds.

In [ ]:
# 12-4


In [ ]:
# 12-5


## Shooting Bullets

Ahora la nave dispara una bala (un rectángulo pequeño) al presionar la barra espaciadora. Las balas viajan en línea recta hacia arriba hasta salir de la pantalla.

### Adding the Bullet Settings

Al final de `__init__()` en `settings.py` se agregan los valores para la clase `Bullet`:

In [ ]:
# settings.py
    def __init__(self):
        # --snip--
        # Bullet settings
        self.bullet_speed = 2.0
        self.bullet_width = 3
        self.bullet_height = 15
        self.bullet_color = (60, 60, 60)

Estos valores crean balas gris oscuro de 3 píxeles de ancho y 15 de alto, un poco más rápidas que la nave.

### Creating the Bullet Class

La clase `Bullet` se guarda en `bullet.py`. Esta es la primera parte:

In [ ]:
# bullet.py
import pygame
from pygame.sprite import Sprite

class Bullet(Sprite):
    """A class to manage bullets fired from the ship."""

    def __init__(self, ai_game):
        """Create a bullet object at the ship's current position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.color = self.settings.bullet_color

        # Create a bullet rect at (0, 0) and then set correct position.
        self.rect = pygame.Rect(0, 0, self.settings.bullet_width,
            self.settings.bullet_height)
        self.rect.midtop = ai_game.ship.rect.midtop

        # Store the bullet's position as a float.
        self.y = float(self.rect.y)

- `Bullet` **hereda de `Sprite`** (de `pygame.sprite`). Con los sprites se pueden agrupar elementos del juego y actuar sobre todos a la vez. Por eso se llama a `super().__init__()`.
- La bala no se basa en una imagen, así que su `rect` se construye desde cero con `pygame.Rect(x, y, ancho, alto)`. Se crea en `(0, 0)` y en la línea siguiente se mueve a su posición real, que depende de la nave: `self.rect.midtop = ai_game.ship.rect.midtop`, para que salga de la parte superior de la nave.
- Se usa un `float` para la coordenada y (`self.y`), para poder ajustar finamente la velocidad.

La segunda parte tiene `update()` y `draw_bullet()`:

In [ ]:
# bullet.py
    def update(self):
        """Move the bullet up the screen."""
        # Update the exact position of the bullet.
        self.y -= self.settings.bullet_speed
        # Update the rect position.
        self.rect.y = self.y

    def draw_bullet(self):
        """Draw the bullet to the screen."""
        pygame.draw.rect(self.screen, self.color, self.rect)

- Al subir por la pantalla, la coordenada y **disminuye**, así que a `self.y` se le resta `bullet_speed`. Luego ese valor se usa para actualizar `self.rect.y`.
- Una vez disparada, la coordenada x nunca cambia: la bala viaja en línea recta aunque la nave se mueva.
- `pygame.draw.rect()` rellena la parte de la pantalla que define el rect de la bala con su color.

### Storing Bullets in a Group

Para manejar las balas ya disparadas, en `AlienInvasion` se crea un grupo: una instancia de `pygame.sprite.Group`, que se comporta como una lista con funciones extra útiles para juegos. Se usa para dibujar las balas y para actualizar sus posiciones en cada vuelta.

In [ ]:
# alien_invasion.py
# --snip--
from ship import Ship
from bullet import Bullet

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        self.ship = Ship(self)
        self.bullets = pygame.sprite.Group()

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            self.ship.update()
            self.bullets.update()
            self._update_screen()
            self.clock.tick(60)

Al llamar a `update()` sobre un grupo, éste llama automáticamente al `update()` de cada sprite que contiene. Así, `self.bullets.update()` llama a `bullet.update()` de cada bala.

### Firing Bullets

Para disparar con la barra espaciadora se modifica `_check_keydown_events()` (no hace falta tocar `_check_keyup_events()`, porque no pasa nada al soltarla). Como disparar requiere varios pasos, se crea un método nuevo, `_fire_bullet()`. También se modifica `_update_screen()` para dibujar cada bala antes de `flip()`.

In [ ]:
# alien_invasion.py
    def _check_keydown_events(self, event):
        # --snip--
        elif event.key == pygame.K_q:
            sys.exit()
        elif event.key == pygame.K_SPACE:
            self._fire_bullet()

    def _check_keyup_events(self, event):
        # --snip--
    def _fire_bullet(self):
        """Create a new bullet and add it to the bullets group."""
        new_bullet = Bullet(self)
        self.bullets.add(new_bullet)

    def _update_screen(self):
        """Update images on the screen, and flip to the new screen."""
        self.screen.fill(self.settings.bg_color)
        for bullet in self.bullets.sprites():
            bullet.draw_bullet()
        self.ship.blitme()

        pygame.display.flip()

# --snip--

- En `_fire_bullet()` se crea una instancia de `Bullet` (`new_bullet`) y se agrega al grupo con `add()`, que es parecido a `append()` pero escrito para los grupos de Pygame.
- `bullets.sprites()` devuelve una lista con todos los sprites del grupo. Se recorre y se llama a `draw_bullet()` en cada uno.
- Ese loop va **antes** de dibujar la nave, para que las balas no queden encima de ella.

Al ejecutarlo, se puede mover la nave y disparar tantas balas como se quiera. Se puede cambiar su tamaño, color y velocidad en `settings.py`.

### Deleting Old Bullets

Las balas "desaparecen" al llegar arriba solo porque Pygame no puede dibujarlas fuera de la pantalla, pero **siguen existiendo**: su coordenada y se vuelve cada vez más negativa. Siguen consumiendo memoria y procesamiento, así que hay que eliminarlas. Se detectan cuando el `bottom` de su rect llega a `0`:

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            self.ship.update()
            self.bullets.update()

            # Get rid of bullets that have disappeared.
            for bullet in self.bullets.copy():
                if bullet.rect.bottom <= 0:
                    self.bullets.remove(bullet)
            print(len(self.bullets))

            self._update_screen()
            self.clock.tick(60)

- Cuando se usa un `for` con una lista (o un grupo), Python espera que ésta **mantenga el mismo largo** mientras el loop corre. Por eso no se pueden eliminar elementos dentro del `for`: se recorre una **copia** del grupo con `copy()`, y así se puede modificar el grupo original.
- Si la bala salió por arriba, se elimina con `remove()`.
- El `print(len(self.bullets))` sirve para comprobar que el número de balas baja a cero después de cada disparo. **Hay que borrarlo después de verificar**: escribir en la terminal tarda más que dibujar los gráficos y el juego se pone lento.

### Limiting the Number of Bullets

Muchos juegos limitan las balas en pantalla para que el jugador apunte mejor. Primero se guarda el límite en `settings.py`:

In [ ]:
# settings.py
        # Bullet settings
        # --snip--
        self.bullet_color = (60, 60, 60)
        self.bullets_allowed = 3

In [ ]:
# alien_invasion.py
    def _fire_bullet(self):
        """Create a new bullet and add it to the bullets group."""
        if len(self.bullets) < self.settings.bullets_allowed:
            new_bullet = Bullet(self)
            self.bullets.add(new_bullet)

Al presionar la barra espaciadora se revisa el largo de `bullets`: solo se crea una bala nueva si hay menos de tres. Si ya hay tres, no pasa nada.

### Creating the _update_bullets() Method

Para mantener ordenada la clase, el código de las balas se mueve a un método propio, `_update_bullets()`, justo antes de `_update_screen()`:

In [ ]:
# alien_invasion.py
    def _update_bullets(self):
        """Update position of bullets and get rid of old bullets."""
        # Update bullet positions.
        self.bullets.update()

        # Get rid of bullets that have disappeared.
        for bullet in self.bullets.copy():
            if bullet.rect.bottom <= 0:
                self.bullets.remove(bullet)

In [ ]:
# alien_invasion.py
        while True:
            self._check_events()
            self.ship.update()
            self._update_bullets()
            self._update_screen()
            self.clock.tick(60)

Ahora el loop principal vuelve a ser corto: se revisa la entrada del jugador, se actualizan la nave y las balas, se dibuja la pantalla y se avanza el reloj.

### Comparison: Shooting Bullets

| | `Ship` | `Bullet` |
|---|---|---|
| Hereda de | nada | `Sprite` |
| Forma | una imagen (`ship.bmp`) | un rectángulo creado con `pygame.Rect()` |
| Cómo se dibuja | `screen.blit(self.image, self.rect)` en `blitme()` | `pygame.draw.rect(self.screen, self.color, self.rect)` en `draw_bullet()` |
| Posición inicial | `rect.midbottom` = `midbottom` de la pantalla | `rect.midtop` = `midtop` de la nave |
| Se mueve | en x (`self.x`) con los flags | en y (`self.y`), disminuyendo, sola |
| Cantidad | una sola instancia | varias, guardadas en un grupo |

| | Lista | `Group` de Pygame |
|---|---|---|
| Agregar | `.append(x)` | `.add(x)` |
| Quitar | `.remove(x)` | `.remove(x)` |
| Recorrer | `for x in lista:` | `for x in grupo.sprites():` |
| Actualizar todos | un `for` que llama a `update()` en cada uno | `grupo.update()` |
| Copia para borrar mientras se recorre | `lista.copy()` o `lista[:]` | `grupo.copy()` |

| | `self.bullets.update()` | `bullet.draw_bullet()` |
|---|---|---|
| Qué hace | actualiza la posición de todas las balas | dibuja una bala |
| Dónde se llama | en `_update_bullets()` | en `_update_screen()`, para cada bala |

### Try it Yourself

> 12-6. Sideways Shooter:
Write a game that places a ship on the left side of the screen and allows the player to move the ship up and down. Make the ship fire a bullet that travels right across the screen when the player presses the spacebar. Make sure bullets are deleted once they disappear off the screen.

In [ ]:
# 12-6


## Final Code

Los archivos del proyecto al terminar el capítulo (se ejecuta solo `alien_invasion.py`; la imagen `ship.bmp` va en la carpeta `images`).

**alien_invasion.py**

In [ ]:
# alien_invasion.py
import sys

import pygame

from settings import Settings
from ship import Ship
from bullet import Bullet


class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.clock = pygame.time.Clock()
        self.settings = Settings()

        self.screen = pygame.display.set_mode(
            (self.settings.screen_width, self.settings.screen_height))
        pygame.display.set_caption("Alien Invasion")

        self.ship = Ship(self)
        self.bullets = pygame.sprite.Group()

    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            self.ship.update()
            self._update_bullets()
            self._update_screen()
            self.clock.tick(60)

    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                self._check_keydown_events(event)
            elif event.type == pygame.KEYUP:
                self._check_keyup_events(event)

    def _check_keydown_events(self, event):
        """Respond to keypresses."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = True
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = True
        elif event.key == pygame.K_q:
            sys.exit()
        elif event.key == pygame.K_SPACE:
            self._fire_bullet()            

    def _check_keyup_events(self, event):
        """Respond to key releases."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = False
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = False

    def _fire_bullet(self):
        """Create a new bullet and add it to the bullets group."""
        if len(self.bullets) < self.settings.bullets_allowed:
            new_bullet = Bullet(self)
            self.bullets.add(new_bullet)

    def _update_bullets(self):
        """Update position of bullets and get rid of old bullets."""
        # Update bullet positions.
        self.bullets.update()

        # Get rid of bullets that have disappeared.
        for bullet in self.bullets.copy():
            if bullet.rect.bottom <= 0:
                self.bullets.remove(bullet)

    def _update_screen(self):
        """Update images on the screen, and flip to the new screen."""
        self.screen.fill(self.settings.bg_color)
        for bullet in self.bullets.sprites():
            bullet.draw_bullet()
        self.ship.blitme()

        pygame.display.flip()


if __name__ == '__main__':
    # Make a game instance, and run the game.
    ai = AlienInvasion()
    ai.run_game()

**settings.py**

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        """Initialize the game's settings."""
        # Screen settings
        self.screen_width = 1200
        self.screen_height = 800
        self.bg_color = (230, 230, 230)

        # Ship settings.
        self.ship_speed = 1.5

        # Bullet settings
        self.bullet_speed = 2.0
        self.bullet_width = 3
        self.bullet_height = 15
        self.bullet_color = (60, 60, 60)
        self.bullets_allowed = 3

**ship.py**

In [ ]:
# ship.py
import pygame


class Ship:
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.screen_rect = ai_game.screen.get_rect()

        # Load the ship image and get its rect.
        self.image = pygame.image.load('images/ship.bmp')
        self.rect = self.image.get_rect()

        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

        # Store a float for the ship's exact horizontal position.
        self.x = float(self.rect.x)

        # Movement flags; start with a ship that's not moving.
        self.moving_right = False
        self.moving_left = False

    def update(self):
        """Update the ship's position based on movement flags."""
        # Update the ship's x value, not the rect.
        if self.moving_right and self.rect.right < self.screen_rect.right:
            self.x += self.settings.ship_speed
        if self.moving_left and self.rect.left > 0:
            self.x -= self.settings.ship_speed
            
        # Update rect object from self.x.
        self.rect.x = self.x

    def blitme(self):
        """Draw the ship at its current location."""
        self.screen.blit(self.image, self.rect)

**bullet.py**

In [ ]:
# bullet.py
import pygame
from pygame.sprite import Sprite

class Bullet(Sprite):
    """A class to manage bullets fired from the ship."""

    def __init__(self, ai_game):
        """Create a bullet object at the ship's current position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.color = self.settings.bullet_color

        # Create a bullet rect at (0, 0) and then set correct position.
        self.rect = pygame.Rect(0, 0, self.settings.bullet_width,
            self.settings.bullet_height)
        self.rect.midtop = ai_game.ship.rect.midtop

        # Store the bullet's position as a float.
        self.y = float(self.rect.y)

    def update(self):
        """Move the bullet up the screen."""
        # Update the exact position of the bullet.
        self.y -= self.settings.bullet_speed
        # Update the rect position.
        self.rect.y = self.y

    def draw_bullet(self):
        """Draw the bullet to the screen."""
        pygame.draw.rect(self.screen, self.color, self.rect)

## Comparison: Chapter 12

| Si necesito... | Uso |
|---|---|
| Crear la ventana del juego | `pygame.display.set_mode((ancho, alto))` |
| Repetir el juego mientras corre | main loop: `while True:` |
| Responder a lo que hace el jugador | `pygame.event.get()` + `event.type` / `event.key` |
| Que el juego corra igual en todos los equipos | `pygame.time.Clock()` + `clock.tick(60)` |
| Guardar todos los valores del juego | clase `Settings` (`settings.py`) |
| Mostrar una imagen | `pygame.image.load()` + `screen.blit()` |
| Ubicar un elemento | los atributos de su `rect` |
| Movimiento continuo | flag (`moving_right`) + `update()` en cada vuelta |
| Velocidad decimal | guardar la posición en un `float` (`self.x`) y copiarla a `rect.x` |
| Que un elemento no se salga de la pantalla | comparar `rect.right` / `rect.left` con los bordes de `screen_rect` |
| Dibujar un rectángulo sin imagen | `pygame.Rect()` + `pygame.draw.rect()` |
| Manejar muchos elementos iguales | `pygame.sprite.Group()` |
| Borrar elementos de un grupo mientras se recorre | recorrer `grupo.copy()` |
| Limitar las balas en pantalla | `len(self.bullets) < self.settings.bullets_allowed` |
| Mantener el código ordenado | refactoring con helper methods (`_nombre`) |

| | `alien_invasion.py` | `settings.py` | `ship.py` | `bullet.py` |
|---|---|---|---|---|
| Clase | `AlienInvasion` | `Settings` | `Ship` | `Bullet` |
| Responsabilidad | main loop, eventos y pantalla | valores del juego | la nave | las balas |
| Se ejecuta | **sí** | no, se importa | no, se importa | no, se importa |